# 文本与数据格式

学习目标：把文本记录解析为经过校验的数据，在 JSON、CSV 与 TOML 之间选择合适的处理方式，并识别转换中的信息损失与 pickle 的信任边界。

前置知识：字符串与原始字符串、列表和字典、函数与类型标注、异常处理、文本与二进制文件、with。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

数据在内存或自动清理的临时目录中处理。

## 1 用正则表达式提取文本

### 1.1 模式、原始字符串与分组

正则表达式（regular expression）用模式描述要匹配的字符。下面从 python:45 提取课程代码和学习分钟数；分组得到的仍是字符串，数值转换是另一项操作。

原始字符串的 r 前缀属于 Python 字符串语法，让反斜杠保留到正则解析阶段；它不会关闭正则自身的特殊语法。字符串章节讲字面值，本章关注 re 如何解释模式。

| 模式写法 | 中文名称／含义 |
| --- | --- |
| [a-z]+ | 一个或多个 ASCII 小写字母；+ 表示前一项重复至少一次 |
| [0-9]{1,3} | 一到三位 ASCII 数字；花括号给出重复次数范围 |
| \d+ | 一个或多个 Unicode 十进制数字，默认不只包含 0 到 9 |
| (?P\<course\>[a-z]+) | 命名捕获组；course 是组名，括号内模式匹配到的文本可按组名取出 |

re.compile 返回可重复使用的模式对象。匹配成功后，group(0) 返回完整匹配，group(1) 返回第一个捕获组；命名组也占据编号，groupdict() 汇总所有命名组。

In [1]:
import re

record_pattern = re.compile(r"(?P<course>[a-z]+):(?P<minutes>[0-9]{1,3})")
match = record_pattern.fullmatch("python:45")
assert match is not None  # 本例应完整匹配；使用分组前先确认成功。
print(match.group(0), match.group(1))  # python:45 python
print(match.groupdict())  # {'course': 'python', 'minutes': '45'}
print(type(match.group("minutes")).__name__)  # str

print(re.fullmatch(r"\d+", "４５") is not None)  # True：全角数字。
print(re.fullmatch(r"[0-9]+", "４５") is not None)  # False

python:45 python
{'course': 'python', 'minutes': '45'}
str
True
False


### 1.2 整段校验与局部搜索

fullmatch 要求整段文本符合模式；search 从左到右寻找第一个匹配位置。找不到时都返回 None，应先判断再调用分组方法。

search 找到一段合法内容，不代表周围文本也合法。检查整个字段时使用 fullmatch，也能拒绝字段末尾多出的换行。

In [2]:
text = "记录 python:45 已完成"
found = record_pattern.search(text)
assert found is not None
print(found.group(0))  # python:45：search 只提取局部。
print(record_pattern.fullmatch(text))  # None：整段文本不符合模式。
print(record_pattern.fullmatch("python:45\n"))  # None：多了换行。
print(record_pattern.search("没有学习记录"))  # None

python:45
None
None
None


### 1.3 遍历匹配与替换

finditer 按从左到右的顺序产出互不重叠的匹配对象。sub 返回替换后的新字符串，默认替换所有互不重叠的匹配，原字符串不变。

替换文本中的 \g\<course\> 引用名为 course 的捕获组，\g\<minutes\> 同理；替换文本也有自己的反斜杠解释规则。下面仍使用上一节的模式。

In [3]:
notes = "python:45; sql:30"
records = [item.groupdict() for item in record_pattern.finditer(notes)]
print(records)
# 两条记录依次对应 python、sql，minutes 此时仍是字符串。
print(record_pattern.sub(r"\g<course> 学习 \g<minutes> 分钟", notes))
# python 学习 45 分钟; sql 学习 30 分钟
print(notes)  # python:45; sql:30：原字符串未改变。

[{'course': 'python', 'minutes': '45'}, {'course': 'sql', 'minutes': '30'}]
python 学习 45 分钟; sql 学习 30 分钟
python:45; sql:30


## 2 JSON 的类型与转换限制

### 2.1 对象与 JSON 文本

序列化（serialization）把对象转换为可保存或传输的表示；反序列化则从这种表示重建对象。JSON 是文本格式，json.dumps 返回 str，json.loads 把 JSON 文本解析为 Python 对象。

下面列出常用内置值的默认转换。JSON 对象的名称只能是字符串；布尔值和空值的拼写与 Python 不同。

| Python 类型或值 | 中文名称／含义 | JSON 表示 | 默认读回 Python |
| --- | --- | --- | --- |
| dict | 字典 | object，对象 | dict |
| list | 列表 | array，数组 | list |
| tuple | 元组 | array，数组 | list |
| str | 字符串 | string，字符串 | str |
| int | 整数 | number，整数写法 | int |
| float | 浮点数 | number，小数或指数写法 | float |
| True | 真值 | true | True |
| False | 假值 | false | False |
| None | 空值 | null | None |

ensure_ascii=False 保留中文等非 ASCII 字符，不影响解析后的值，也不负责文件编码；默认值 True 会把这些字符转义。indent=2 用两个空格缩进，便于阅读。

In [4]:
import json

study_record = {"course": "python", "minutes": 45, "note": "复习分组"}
json_text = json.dumps(study_record, ensure_ascii=False, indent=2)
print(json_text)  # 中文直接可读；这是 JSON 文本，不是字典。
print(json.loads(json_text) == study_record)  # True
print(json.dumps("中文"))  # "\u4e2d\u6587"：默认转义。
print(json.loads('[true, false, null, 45, 1.5]'))
# [True, False, None, 45, 1.5]：大小写与空值被转换。

{
  "course": "python",
  "minutes": 45,
  "note": "复习分组"
}
True
"\u4e2d\u6587"
[True, False, None, 45, 1.5]


### 2.2 字符串接口与流接口

| API | 中文名称／含义 |
| --- | --- |
| json.dumps | 把对象编码为 JSON 字符串 |
| json.loads | 从 JSON 字符串等输入解析对象 |
| json.dump | 把 JSON 文本写入支持 write 的流 |
| json.load | 从支持 read 的流读取并解析 JSON 文档 |

JSON 输出是文本，写文件时显式指定 UTF-8。下面用 StringIO 演示同样的流接口；一个 JSON 文档写入一个完整值，不能反复 dump 多个独立值并直接拼成一份文档。

In [5]:
from io import StringIO

with StringIO() as stream:
    json.dump([study_record], stream, ensure_ascii=False)
    stream.seek(0)
    restored_records = json.load(stream)

print(restored_records == [study_record])  # True：完整列表作为一个值。

True


### 2.3 往返转换可能丢失信息

tuple 与 list 都编码成 JSON 数组，读回后不能据此区分原类型。编码器还接受 int、float、bool、None 作为字典键，但会把键转换为字符串；其他键类型默认引发 TypeError。

非字符串键可能与已有字符串键变成同名。Python 的 JSON 解码器默认只保留同名成员的最后一个值，因此应在输出前约定字符串键，并避免名称重复。

In [6]:
original = {1: (45, 30)}
restored = json.loads(json.dumps(original))
print(restored)  # {'1': [45, 30]}：键类型和元组类型都没有保留。
print(original == restored)  # False

collision_text = json.dumps({1: "整数键", "1": "字符串键"}, ensure_ascii=False)
print(collision_text)  # 两个成员名称现在都是 "1"。
print(json.loads(collision_text))  # {'1': '字符串键'}：前一个值丢失。

{'1': [45, 30]}
False
{"1": "整数键", "1": "字符串键"}
{'1': '字符串键'}


### 2.4 不支持的对象需要明确转换

JSON 默认编码器不能直接处理 set、bytes 或普通自定义类实例。需要根据字段含义，先转换为它支持的值；例如集合可以约定为数组，但顺序和去重含义需由应用自己约定。

下面只观察不支持的类型与键，不使用自动转成字符串的回退，以免掩盖数据含义。

In [7]:
for unsupported in ({45, 30}, b"python", {(1, 2): "pair"}):
    try:
        json.dumps(unsupported)
    except TypeError as error:
        print(type(error).__name__)  # 三次 TypeError。
    else:
        raise AssertionError("本例应拒绝不支持的对象或键")

TypeError
TypeError
TypeError


### 2.5 写出时拒绝非有限浮点值

NaN 表示非数，Infinity 与 -Infinity 表示正、负无穷；JSON 标准不接受它们。Python 的 json 模块默认仍会输出这些扩展写法。

需要符合 JSON 标准的数值输出时，设置 allow_nan=False；遇到非有限浮点值会抛出 ValueError。这个参数只控制编码端的这项行为，不执行字段或业务校验。

In [8]:
non_finite_values = [float("nan"), float("inf"), float("-inf")]
print(json.dumps(non_finite_values))  # [NaN, Infinity, -Infinity]

for value in non_finite_values:
    try:
        json.dumps({"minutes": value}, allow_nan=False)
    except ValueError as error:
        print(type(error).__name__)  # 每种非有限值都应触发 ValueError。
    else:
        raise AssertionError("非有限值不应被写入本例的 JSON")

[NaN, Infinity, -Infinity]
ValueError
ValueError
ValueError


### 2.6 读取时明确拒绝扩展数值

json.loads 默认也接受 NaN、Infinity 和 -Infinity。parse_constant 可以接收一个函数，解析器读到这三个名称时会把名称交给它；函数主动抛出异常即可拒绝。

这只拦截这三个扩展名称，并非完整的数值范围校验。业务要求限定区间时，仍须检查解析后的值。

In [9]:
def reject_json_constant(name: str) -> None:
    """拒绝 JSON 标准之外的三个非有限数值名称。"""
    raise ValueError(f"JSON 不接受扩展数值：{name}")


print(json.loads("Infinity"))  # inf：默认解码器接受扩展。
for text in ("NaN", "Infinity", "-Infinity"):
    try:
        json.loads(text, parse_constant=reject_json_constant)
    except ValueError as error:
        print(str(error))  # 分别指出被拒绝的扩展名称。
    else:
        raise AssertionError("本例应拒绝扩展数值")

inf
JSON 不接受扩展数值：NaN
JSON 不接受扩展数值：Infinity
JSON 不接受扩展数值：-Infinity


## 3 CSV 的字段与类型约定

### 3.1 让 CSV 模块处理分隔符与引号

CSV 用行和字段表达表格。字段本身可能包含逗号、双引号甚至换行，直接按逗号或换行拆字符串会破坏这些字段。

csv.writer 默认只给需要的字段加引号，并把字段内部的双引号写成两个双引号；csv.reader 按对应规则读取。读写文件时使用 newline=""，把换行处理交给 csv，避免字段内换行解释错误或写出多余的回车。文本编码仍单独指定。

In [10]:
import csv
from pathlib import Path
from tempfile import TemporaryDirectory

rows = [
    ["course", "note"],
    ["python", '复习 "分组", JSON\n再练一次'],
]
with TemporaryDirectory() as directory:
    csv_path = Path(directory) / "notes.csv"
    with csv_path.open("w", encoding="utf-8", newline="") as stream:
        csv.writer(stream).writerows(rows)
    with csv_path.open(encoding="utf-8", newline="") as stream:
        read_rows = list(csv.reader(stream))
    print(read_rows == rows)  # True：引号、逗号、字段内换行均保留。
    print(repr(read_rows[1][1]))  # 换行属于一个字段，不是另一条记录。

print(csv_path.exists())  # False：临时文件已清理。

True
'复习 "分组", JSON\n再练一次'
False


### 3.2 空字符串与类型转换

默认 csv.reader 返回字符串字段，不自动把数字文本变成 int。默认 writer 把 None 写成空字符串，把其他非字符串值通过 str 转换；因此 None 与原本的空字符串往返后无法区分。

需要空值和数值时，先约定每列含义，再显式转换。下面展示信息损失，稍后的综合例子会约定空备注表示 None。

In [11]:
with StringIO(newline="") as stream:
    csv.writer(stream).writerow(["python", 45, None, ""])
    stream.seek(0)
    fields = next(csv.reader(stream))

print(fields)  # ['python', '45', '', '']
print(type(fields[1]).__name__, int(fields[1]))  # str 45
print(fields[2] == fields[3])  # True：无法还原哪一项原本是 None。

['python', '45', '', '']
str 45
True


### 3.3 按表头读写字典

DictReader 未传 fieldnames 时用第一行作为键名；DictWriter 则必须指定 fieldnames，它还决定输出列的顺序，writeheader 写出表头。

DictReader 不负责保证列数一致。默认情况下，少列会把缺失值补为 None，多列会把多出的字段列表放到 None 键下；这与实际存在但内容为空的字段不同。

In [12]:
with StringIO(newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=["course", "minutes", "note"])
    writer.writeheader()
    writer.writerow({"course": "python", "minutes": 45, "note": ""})
    stream.seek(0)
    print(next(csv.DictReader(stream)))
    # {'course': 'python', 'minutes': '45', 'note': ''}

with StringIO("course,note\npython,\nsql\nweb,x,extra\n") as stream:
    parsed_rows = list(csv.DictReader(stream))
print(parsed_rows[0]["note"] == "")  # True：有此列，内容为空。
print(parsed_rows[1]["note"] is None)  # True：整列缺失。
print(parsed_rows[2][None])  # ['extra']：出现多余列。

{'course': 'python', 'minutes': '45', 'note': ''}
True
True
['extra']


### 3.4 默认读取与严格解析的边界

csv 的 strict 参数默认为 False。下面的输入只有一个带起始引号的字段，但到结尾仍未闭合；默认 csv.reader 会读出这个字段。设置 strict=True 后，同一输入会触发 csv.Error。DictReader 也会把 strict 参数传给底层 reader。

严格解析不检查每条记录的字段数是否一致，也不检查字段类型；前述缺列、多列和字符串字段的规则仍然适用。应用需要固定列数、数值类型或取值范围时，仍须自行校验。

In [13]:
unclosed_field = '"未闭合'
with StringIO(unclosed_field, newline="") as stream:
    print(list(csv.reader(stream)))  # [['未闭合']]：默认 strict=False，仍读出字段。

with StringIO(unclosed_field, newline="") as stream:
    try:
        list(csv.reader(stream, strict=True))
    except csv.Error as error:
        print(type(error).__name__)  # Error：严格解析拒绝同一输入。
    else:
        raise AssertionError("strict=True 应拒绝未闭合引号")

[['未闭合']]
Error


## 4 用 tomllib 读取配置

### 4.1 从 TOML 文本读取

TOML 用键值对表达配置，[study] 这样的表头把后续键归入 study 表。字符串加引号，整数直接书写，布尔值使用小写 true 或 false，数组用方括号包含各项。

Python 3.12 的 tomllib 解析 TOML 1.0.0，只有读取接口，不提供写出 TOML 的功能。loads 接收 str，返回 dict；表映射为字典，数组映射为列表。这里讲格式读取，命令行配置的组织留在相应章节。

In [14]:
import tomllib

settings_text = """[study]
course = "python"
minutes = 45
enabled = true
tags = ["基础", "练习"]
"""
settings = tomllib.loads(settings_text)
print(settings["study"]["minutes"])  # 45，已经是 int。
print(settings["study"]["enabled"])  # True，已经是 bool。
print(settings["study"]["tags"])  # ['基础', '练习']

45
True
['基础', '练习']


### 4.2 load 需要二进制文件

tomllib.load 接收可读的二进制文件，应使用 rb 打开；它与接收文本字符串的 loads 不同。下面将上一节的固定文本写入临时文件，再按二进制读取。

无效的 TOML 文档会引发 tomllib.TOMLDecodeError；格式正确也不会自动检查课程名称或分钟数是否满足应用要求。

In [15]:
with TemporaryDirectory() as directory:
    settings_path = Path(directory) / "study.toml"
    settings_path.write_text(settings_text, encoding="utf-8")
    with settings_path.open("rb") as stream:
        file_settings = tomllib.load(stream)
    print(file_settings == settings)  # True：两种读取接口结果相同。

try:
    tomllib.loads("minutes =")
except tomllib.TOMLDecodeError as error:
    print(type(error).__name__)  # TOMLDecodeError：等号后缺少值。
else:
    raise AssertionError("不完整的 TOML 键值对应被拒绝")

True
TOMLDecodeError


## 5 区分解析与业务校验

### 5.1 解析器检查格式，不决定业务规则

json.loads 把文档转成 Python 值；不合法的 JSON 文本会触发 JSONDecodeError。但一个合法的负数，或作为根值的列表，都可能不符合某个应用要求。

本章自行约定一条学习记录的格式：只包含 course、minutes、note；course 是非空 ASCII 小写字母串，minutes 是 1 到 1440 的整数分钟数，note 是 None 或非空字符串。1440 是本例规定的单条上限，其他应用应按自己的规则设定。

In [16]:
parsed_record = json.loads('{"course": "python", "minutes": -5, "note": null}')
print(parsed_record["minutes"])  # -5：格式正确，解析成功不等于记录有效。

try:
    json.loads('{"course": "python",}')
except json.JSONDecodeError as error:
    print(type(error).__name__)  # JSONDecodeError：末尾逗号不合法。
else:
    raise AssertionError("本例的 JSON 语法错误应被发现")

-5
JSONDecodeError


### 5.2 在输入边界检查结构、类型与范围

先检查根值和字段，再检查各字段类型与取值。输入不符合本例约定时，函数抛出带字段说明的 ValueError，而不是返回一条貌似成功的记录。

bool 是 int 的子类，因此只用 isinstance(minutes, int) 会接纳 True。本例用 type(minutes) is int 限定普通整数，再检查范围；类型标注用于描述接口，实际校验由条件语句执行。断言只用于示例核对，不代替输入检查。

In [17]:
def validate_study_record(record: object) -> dict[str, str | int | None]:
    """校验一条学习记录，返回字段固定的新字典；不符合约定时抛出 ValueError。"""
    # 1. 在读取字段前确认结构，拒绝缺失或多余字段。
    if not isinstance(record, dict):
        raise ValueError("学习记录必须是对象")
    if set(record) != {"course", "minutes", "note"}:
        raise ValueError("学习记录必须且只能含 course、minutes、note")

    # 2. 按本例约定检查类型、完整字段格式与范围。
    course = record["course"]
    minutes = record["minutes"]
    note = record["note"]
    if not isinstance(course, str) or re.fullmatch(r"[a-z]+", course) is None:
        raise ValueError("course 必须是非空 ASCII 小写字母串")
    if type(minutes) is not int or not 1 <= minutes <= 1440:
        raise ValueError("minutes 必须是 1 到 1440 的整数，不能是 bool")
    if note is not None and (not isinstance(note, str) or note == ""):
        raise ValueError("note 必须是 None 或非空字符串")
    return {"course": course, "minutes": minutes, "note": note}


print(validate_study_record(study_record) == study_record)  # True

True


### 5.3 解析成功后仍要拒绝不合约定的值

下面的 JSON 都可被解析，但分别违反分钟数范围、字段类型或根值结构。沿用上一节的校验函数，观察错误发生在业务检查阶段。

把本例的所有边界错误统一为 ValueError 是接口选择，并不是 JSON 模块的默认行为。

In [18]:
invalid_records = [
    '{"course": "python", "minutes": -5, "note": null}',
    '{"course": "python", "minutes": true, "note": null}',
    '{"course": "python", "minutes": "45", "note": null}',
    '[]',
]
for text in invalid_records:
    parsed = json.loads(text)  # 每次解析都应成功。
    try:
        validate_study_record(parsed)
    except ValueError as error:
        print(str(error))  # 前三项指出 minutes；最后一项指出根值应为对象。
    else:
        raise AssertionError("本例的不合约定记录应被拒绝")

minutes 必须是 1 到 1440 的整数，不能是 bool


minutes 必须是 1 到 1440 的整数，不能是 bool
minutes 必须是 1 到 1440 的整数，不能是 bool
学习记录必须是对象


## 6 将 CSV 转为经过校验的 JSON

本例要求 CSV 表头按 course、minutes、note 排列，每行都恰好三列；minutes 只接受 ASCII 数字，note 的空字段表示 None。因此空备注没有第二种独立含义，转换约定不会声称保留 None 与空字符串的区别。

读取时启用 strict=True，拒绝未闭合引号等解析错误，并让 csv.Error 原样传出。它不检查表头、列数或字段类型；这些仍由本例显式校验，不符合应用约定时抛出 ValueError。

处理顺序是解析 CSV、检查结构、转换字段、校验业务值，最后编码整个记录列表。前面定义的 validate_study_record 复用于此处；数字转换前的 fullmatch 还会拒绝 int 本身能接受的空白或正负号等写法。表头符合约定但没有数据行时，结果为 JSON 空数组。

In [19]:
def csv_to_json(text: str) -> str:
    """按本章学习记录约定，把 CSV 文本转换为 JSON 数组文本。

    严格解析检测到的 CSV 错误保留 csv.Error，应用约定错误抛出 ValueError。
    """
    records = []
    # 1. 严格解析，并显式检查表头和列数，不把缺失列误当作空备注。
    with StringIO(text, newline="") as stream:
        reader = csv.DictReader(stream, strict=True)
        if reader.fieldnames != ["course", "minutes", "note"]:
            raise ValueError("CSV 表头必须依次为 course、minutes、note")
        for row in reader:
            if None in row or any(value is None for value in row.values()):
                raise ValueError("CSV 每条记录必须恰好有三列")
            # 2. 按列约定转换，再复用统一的记录校验。
            if re.fullmatch(r"[0-9]+", row["minutes"]) is None:
                raise ValueError("CSV minutes 必须只含 ASCII 数字")
            record = {
                "course": row["course"],
                "minutes": int(row["minutes"]),
                "note": None if row["note"] == "" else row["note"],
            }
            records.append(validate_study_record(record))
    # 3. 列表作为一个 JSON 值输出，保留中文并拒绝非有限值。
    return json.dumps(records, ensure_ascii=False, allow_nan=False, indent=2)


csv_text = 'course,minutes,note\npython,45,"复习分组,JSON"\nsql,30,\n'
converted_json = csv_to_json(csv_text)
print(converted_json)
# 两条记录按原顺序输出；minutes 为整数，sql 的 note 为 null。
# python 的备注含逗号，仍然是一个完整字符串字段。

invalid_csv = 'course,minutes,note\npython,45,"未闭合'
try:
    csv_to_json(invalid_csv)
except csv.Error as error:
    print(type(error).__name__)  # Error：未闭合引号被拒绝，原始 csv.Error 传出。
else:
    raise AssertionError("转换函数应拒绝未闭合引号，不能生成 JSON")

[
  {
    "course": "python",
    "minutes": 45,
    "note": "复习分组,JSON"
  },
  {
    "course": "sql",
    "minutes": 30,
    "note": null
  }
]
Error


## 7 了解 pickle 的可信数据要求

pickle 用二进制格式保存 Python 对象结构，支持的类型比 JSON 多，例如 tuple 和 set。它主要服务于 Python 对象恢复；不能因此把它当成通用的跨语言交换格式。

反序列化 pickle 数据可能执行任意代码。只读取来源可信且未被篡改的数据，不加载来历不明的文件或网络字节；扩展名正确并不构成信任依据。下面仅反序列化当前代码刚刚在内存中生成的字节，不接收外部输入。

pickle.dumps 返回 bytes，pickle.loads 从字节恢复对象。JSON 的类型限制与 pickle 的信任要求是两类不同边界，选择格式时都要考虑。

In [20]:
import pickle

trusted_record = {"durations": (45, 30), "courses": {"python", "sql"}}
trusted_bytes = pickle.dumps(trusted_record)
restored_record = pickle.loads(trusted_bytes)
print(type(trusted_bytes).__name__)  # bytes
print(restored_record == trusted_record)  # True
print(type(restored_record["durations"]).__name__)  # tuple
print(type(restored_record["courses"]).__name__)  # set
# 本例的字节在当前进程内生成；不要替换为不可信的外部数据。

bytes
True
tuple
set


## 本章小结

（1）fullmatch 检查整段字段，search 提取局部，finditer 遍历匹配，sub 返回替换后的文本；分组得到的值仍需按业务转换。

（2）JSON 只直接表达部分 Python 类型，tuple 和非字符串键可能在往返后改变；非有限数值需要分别约束编码与解码。

（3）CSV 默认字段是字符串，None 与空字符串会合并；表头、列数、数值和空值含义需要明确约定。

（4）tomllib.loads 读文本，tomllib.load 读二进制流，模块不写 TOML。pickle 只用于可信且未被篡改的数据。

（5）解析器按所选参数处理格式，解析成功不等于业务有效；CSV 的 strict=True 不替代列数与类型校验。自查：转换后哪些值和类型必须保留，哪些变化已经由输入约定说明？

## 练习

（1）先预测下面两个输出，包括字典键和数组元素的类型，再执行核对。解释为什么 search 能找到匹配，以及哪些 Python 类型信息没有保留。

In [21]:
exercise_match = record_pattern.search("备注 sql:30 完成")
if exercise_match is not None:
    print(exercise_match.groupdict())
print(json.loads(json.dumps({2: (True, None, 30)})))
# 运行前记录预测；运行后按键类型、容器类型和元素类型分别核对。

{'course': 'sql', 'minutes': '30'}
{'2': [True, None, 30]}


（2）编写 read_minutes(text)，从 TOML 文本读取 study 表中的 minutes，要求它是 1 到 1440 的普通整数；返回该整数，缺少表或字段以及值不符合约定时抛出 ValueError。语法错误保留 TOMLDecodeError。

分别检查 45、0、true、缺少 minutes、缺少 study 和语法错误六种输入。45 应返回 45；其余必须失败，并区分解析阶段和业务阶段。不要使用 assert 代替函数内的输入检查。

In [22]:
exercise_toml_inputs = [
    "[study]\nminutes = 45",
    "[study]\nminutes = 0",
    "[study]\nminutes = true",
    '[study]\ncourse = "python"',
    "minutes = 45",
    "[study]\nminutes =",
]
# 在此实现 read_minutes，并对每个输入执行有明确异常类型的检查。
# TOMLDecodeError 是 ValueError 的子类；区分阶段时先检查更具体的类型。

（3）编写 json_to_csv(text)，把本章 JSON 记录数组转回 CSV。先确认根值为列表，再逐条调用 validate_study_record；用 DictWriter 写出固定表头并保留记录顺序，None 备注写为空字段。

用下面的记录检查逗号、双引号和字段内换行是否保留。再把返回的 CSV 传给 csv_to_json，解析所得 JSON 后应与输入记录相等；另外测试非列表根值和 minutes 为 true 的记录均被拒绝。整个过程使用 StringIO，不留下文件。

In [23]:
exercise_records = [
    {"course": "python", "minutes": 45, "note": '复习 "分组", JSON\n继续'},
    {"course": "sql", "minutes": 30, "note": None},
]
exercise_json = json.dumps(exercise_records, ensure_ascii=False)
# 在此实现 json_to_csv，再用 csv_to_json 和 json.loads 检查往返结果。
# 核对空备注、字段中的标点与换行，以及记录的先后顺序。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [re：原始字符串](https://docs.python.org/3.12/library/re.html#raw-string-notation)、[正则语法、分组和 Unicode 数字](https://docs.python.org/3.12/library/re.html#regular-expression-syntax)；[compile](https://docs.python.org/3.12/library/re.html#re.compile)、[fullmatch](https://docs.python.org/3.12/library/re.html#re.fullmatch)、[search](https://docs.python.org/3.12/library/re.html#re.search)、[finditer](https://docs.python.org/3.12/library/re.html#re.finditer)、[sub 与替换分组引用](https://docs.python.org/3.12/library/re.html#re.sub)、[group](https://docs.python.org/3.12/library/re.html#re.Match.group)、[groupdict](https://docs.python.org/3.12/library/re.html#re.Match.groupdict)；[JSON 编码类型](https://docs.python.org/3.12/library/json.html#json.JSONEncoder)、[解码类型](https://docs.python.org/3.12/library/json.html#json.JSONDecoder)、[dump 的流、转义和非有限值参数](https://docs.python.org/3.12/library/json.html#json.dump)、[dumps 与键转换](https://docs.python.org/3.12/library/json.html#json.dumps)、[load 与 parse_constant](https://docs.python.org/3.12/library/json.html#json.load)、[loads](https://docs.python.org/3.12/library/json.html#json.loads)、[非有限数值](https://docs.python.org/3.12/library/json.html#infinite-and-nan-number-values)、[同名成员](https://docs.python.org/3.12/library/json.html#repeated-names-within-an-object)、[JSONDecodeError](https://docs.python.org/3.12/library/json.html#json.JSONDecodeError)；[CSV reader](https://docs.python.org/3.12/library/csv.html#csv.reader)、[writer 的 None 与字符串转换](https://docs.python.org/3.12/library/csv.html#csv.writer)、[DictReader 的缺列和多列](https://docs.python.org/3.12/library/csv.html#csv.DictReader)、[strict 的默认值与解析异常](https://docs.python.org/3.12/library/csv.html#csv.Dialect.strict)、[DictWriter](https://docs.python.org/3.12/library/csv.html#csv.DictWriter)、[writeheader](https://docs.python.org/3.12/library/csv.html#csv.DictWriter.writeheader)、[writerows](https://docs.python.org/3.12/library/csv.html#csv.csvwriter.writerows)、[最小引号](https://docs.python.org/3.12/library/csv.html#csv.QUOTE_MINIMAL)、[双引号转义](https://docs.python.org/3.12/library/csv.html#csv.Dialect.doublequote)、[newline 脚注 1](https://docs.python.org/3.12/library/csv.html#id4)；[tomllib 的只读范围](https://docs.python.org/3.12/library/tomllib.html#module-tomllib)、[load](https://docs.python.org/3.12/library/tomllib.html#tomllib.load)、[loads](https://docs.python.org/3.12/library/tomllib.html#tomllib.loads)、[类型映射](https://docs.python.org/3.12/library/tomllib.html#conversion-table)、[TOMLDecodeError](https://docs.python.org/3.12/library/tomllib.html#tomllib.TOMLDecodeError)；[pickle 的信任警告](https://docs.python.org/3.12/library/pickle.html#module-pickle)、[与 JSON 比较](https://docs.python.org/3.12/library/pickle.html#comparison-with-json)、[可序列化类型](https://docs.python.org/3.12/library/pickle.html#what-can-be-pickled-and-unpickled)、[dumps](https://docs.python.org/3.12/library/pickle.html#pickle.dumps)、[loads](https://docs.python.org/3.12/library/pickle.html#pickle.loads)；[bool 与 int 的关系](https://docs.python.org/3.12/library/stdtypes.html#boolean-type-bool)、[type](https://docs.python.org/3.12/library/functions.html#type)、[isinstance](https://docs.python.org/3.12/library/functions.html#isinstance)、[int 的字符串输入](https://docs.python.org/3.12/library/functions.html#int)；[StringIO](https://docs.python.org/3.12/library/io.html#io.StringIO)、[TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)、[Path.open](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.open)、[Path.write_text](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.write_text)。业务字段、分钟数范围和空备注约定是本章示例的应用规则。 |
| TOML 官方规范（1.0.0） | [键值对与值类型](https://toml.io/en/v1.0.0#keyvalue-pair)、[表](https://toml.io/en/v1.0.0#table)、[数组](https://toml.io/en/v1.0.0#array)、[整数](https://toml.io/en/v1.0.0#integer)、[布尔值](https://toml.io/en/v1.0.0#boolean)，用于配置语法与 TOML 值含义。 |